# Section 05: Managing Implementation

**Companion notebook for**: `05-managing-implementation.html`

## Learning Objectives
- Write Terraform configurations for GCP resources
- Configure Apigee API proxies
- Implement deployment strategies (canary, blue/green)
- Work with Cloud SDKs and client libraries
- Use Gemini Cloud Assist patterns

## Prerequisites
- A Google Cloud project with billing enabled
- Python 3.10+

---
## Setup

In [ ]:
%pip install -q google-cloud-storage google-cloud-bigquery google-cloud-run vertexai

In [ ]:
import os, json
PROJECT_ID = os.environ.get('GCP_PROJECT_ID', 'your-project-id')
REGION = 'us-central1'
print(f'Project: {PROJECT_ID}\nRegion: {REGION}')
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

---
## Section 1: Terraform Patterns

Generate production-ready Terraform configurations for common GCP patterns.

In [ ]:
terraform_configs = {
    'Cloud SQL with HA': '''resource "google_sql_database_instance" "primary" {
  name             = "prod-db"
  database_version = "POSTGRES_15"
  region           = "us-central1"

  settings {
    tier              = "db-custom-4-16384"
    availability_type = "REGIONAL"  # HA

    backup_configuration {
      enabled                        = true
      point_in_time_recovery_enabled = true
      transaction_log_retention_days = 7
      backup_retention_settings {
        retained_backups = 30
      }
    }

    ip_configuration {
      ipv4_enabled    = false
      private_network = google_compute_network.vpc.id
      require_ssl     = true
    }

    disk_autoresize = true
    disk_size       = 100
    disk_type       = "PD_SSD"
  }

  deletion_protection = true
}''',
    'Cloud Run Service': '''resource "google_cloud_run_v2_service" "api" {
  name     = "my-api"
  location = "us-central1"

  template {
    containers {
      image = "us-central1-docker.pkg.dev/PROJECT/repo/api:latest"
      resources {
        limits = { cpu = "2", memory = "1Gi" }
      }
      env { name = "DB_HOST", value = "10.0.2.5" }
    }

    scaling {
      min_instance_count = 1
      max_instance_count = 100
    }

    vpc_access {
      connector = google_vpc_access_connector.connector.id
      egress    = "PRIVATE_RANGES_ONLY"
    }

    service_account = google_service_account.run_sa.email
  }

  traffic {
    percent = 100
    type    = "TRAFFIC_TARGET_ALLOCATION_TYPE_LATEST"
  }
}''',
}

for name, config in terraform_configs.items():
    print(f'## {name}')
    print(config)
    print('\n' + '=' * 60 + '\n')

---
## Section 2: Terraform State Management

In [ ]:
state_config = '''# Backend configuration for team collaboration
terraform {
  backend "gcs" {
    bucket = "my-org-tf-state"
    prefix = "terraform/production"
  }
}

# State management best practices:
# 1. NEVER store state locally in production
# 2. Enable versioning on the GCS state bucket
# 3. Use separate state files per environment
# 4. Lock state during operations (GCS does this automatically)
# 5. Encrypt state bucket with CMEK for sensitive data
'''
print(state_config)

print('State bucket setup commands:')
cmds = [
    'gsutil mb -l us-central1 gs://my-org-tf-state',
    'gsutil versioning set on gs://my-org-tf-state',
    'gsutil lifecycle set lifecycle.json gs://my-org-tf-state',
]
for cmd in cmds:
    print(f'  {cmd}')

---
## Section 3: Apigee API Management

In [ ]:
apigee_comparison = [
    ('Apigee X', 'Google-managed, VPC-peered', 'Enterprise API programs', 'Full features, global'),
    ('Apigee Hybrid', 'Runtime on your K8s', 'Data residency, on-prem backends', 'Runtime in your cluster'),
    ('API Gateway', 'Serverless', 'Simple APIs, Cloud Run/Functions', 'Lightweight, OpenAPI-based'),
]

print(f"{'Platform':<16} {'Deployment':<30} {'Best For':<35} {'Key Difference'}")
print('-' * 100)
for p in apigee_comparison:
    print(f'{p[0]:<16} {p[1]:<30} {p[2]:<35} {p[3]}')

print('\nApigee Policies:')
policies = [
    ('OAuth 2.0', 'Authentication and authorization'),
    ('Quota', 'Per-consumer rate limiting'),
    ('Spike Arrest', 'Per-proxy rate limiting'),
    ('Response Cache', 'Cache backend responses'),
    ('JSON-to-XML', 'Message transformation'),
    ('CORS', 'Cross-origin resource sharing'),
]
for name, desc in policies:
    print(f'  {name:<20} {desc}')

---
## Section 4: Deployment Strategy Comparison

In [ ]:
strategies = [
    ('Rolling', 'Gradual replace', 'Medium', 'Medium', 'GKE, MIGs'),
    ('Blue/Green', 'Parallel env, switch', 'Low', 'Instant', 'Cloud Run, GKE'),
    ('Canary', 'Small % to new', 'Lowest', 'Instant', 'Cloud Deploy, Cloud Run'),
    ('Recreate', 'Kill old, start new', 'Highest', 'Slow', 'Basic K8s'),
    ('A/B Testing', 'Route by attributes', 'Low', 'Instant', 'Istio, Cloud Run'),
]

print(f"{'Strategy':<14} {'How':<25} {'Risk':<10} {'Rollback':<10} {'GCP Support'}")
print('-' * 75)
for s in strategies:
    print(f'{s[0]:<14} {s[1]:<25} {s[2]:<10} {s[3]:<10} {s[4]}')

print('\nCloud Run Canary Deployment:')
print('  # Deploy new revision')
print('  gcloud run deploy my-api --image=gcr.io/PROJECT/api:v2 --no-traffic')
print('  # Route 10% to new revision')
print('  gcloud run services update-traffic my-api --to-revisions=LATEST=10')
print('  # Promote to 100%')
print('  gcloud run services update-traffic my-api --to-latest')
print('  # Rollback')
print('  gcloud run services update-traffic my-api --to-revisions=my-api-00001=100')

---
## Section 5: Cloud SDK Client Libraries

In [ ]:
# Cloud Storage client library example
from google.cloud import storage

try:
    client = storage.Client(project=PROJECT_ID)
    buckets = list(client.list_buckets(max_results=5))
    print(f'Found {len(buckets)} buckets:')
    for b in buckets:
        print(f'  - {b.name} (location: {b.location}, class: {b.storage_class})')
except Exception as e:
    print(f'Storage API failed: {e}')
    print('\nMock output:')
    for name, loc, cls in [('data-lake', 'US', 'STANDARD'), ('backups', 'US-CENTRAL1', 'NEARLINE'), ('logs', 'US', 'COLDLINE')]:
        print(f'  - {name} (location: {loc}, class: {cls})')

In [ ]:
# BigQuery client library example
from google.cloud import bigquery

try:
    client = bigquery.Client(project=PROJECT_ID)
    datasets = list(client.list_datasets(max_results=5))
    print(f'Found {len(datasets)} datasets:')
    for d in datasets:
        print(f'  - {d.dataset_id}')
except Exception as e:
    print(f'BigQuery API failed: {e}')
    print('\nMock output:')
    for ds in ['analytics', 'ml_features', 'audit_logs']:
        print(f'  - {ds}')

---
## Section 6: Gemini Cloud Assist Patterns

In [ ]:
# Gemini Cloud Assist use cases
assist_cases = [
    ('Troubleshooting', 'Why is my Cloud Run service returning 502 errors?', 'Analyzes logs, metrics, and config to diagnose issues'),
    ('Code Generation', 'Generate a Terraform config for a Cloud SQL HA instance', 'Produces HCL code with best practices'),
    ('Cost Analysis', 'Which VMs in project-x are over-provisioned?', 'Queries Recommender insights and summarizes'),
    ('Security Review', 'Are there any public buckets in my organization?', 'Scans SCC findings and org policies'),
    ('Architecture', 'What is the best way to migrate this MySQL DB?', 'Recommends DMS, Cloud SQL, migration steps'),
]

print('Gemini Cloud Assist Use Cases')
print('=' * 80)
for category, question, capability in assist_cases:
    print(f'\n  Category: {category}')
    print(f'  Question: "{question}"')
    print(f'  What it does: {capability}')

print('\nKey point: Gemini Cloud Assist accelerates implementation but does not replace architecture judgment.')

---
## Section 7: Binary Authorization

In [ ]:
print('Binary Authorization Flow')
print('=' * 50)
print('''
1. Developer pushes code --> Cloud Build
2. Cloud Build builds image --> Artifact Registry
3. Artifact Registry scans for CVEs
4. If scan passes --> Attestor signs the image
5. GKE admission controller checks signature
6. Only signed images are allowed to deploy

Commands:
  # Create attestor
  gcloud container binauthz attestors create my-attestor \\
      --attestation-authority-note=my-note \\
      --attestation-authority-note-project=PROJECT_ID

  # Create attestation
  gcloud container binauthz attestations sign-and-create \\
      --artifact-url=us-central1-docker.pkg.dev/PROJECT/repo/app@sha256:abc123 \\
      --attestor=my-attestor \\
      --keyversion=projects/PROJECT/locations/global/keyRings/my-ring/cryptoKeys/my-key/cryptoKeyVersions/1

  # Set GKE policy to require attestation
  gcloud container binauthz policy import policy.yaml
''')

---
## Summary

This notebook covered PCA Section 05:

1. **Terraform** -- Cloud SQL HA and Cloud Run configurations
2. **State Management** -- GCS backend, versioning, best practices
3. **Apigee** -- X vs Hybrid vs API Gateway comparison, policies
4. **Deployment Strategies** -- Rolling, Blue/Green, Canary comparison
5. **Cloud SDKs** -- Storage and BigQuery client library examples
6. **Gemini Cloud Assist** -- Use cases and capabilities
7. **Binary Authorization** -- Container supply chain security

**Next**: Section 06 covers operations excellence.